In [ ]:
%pip install plotly scipy numpy

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go

def nbody_rhs(t, state, m, G=1.0, eps=1e-3):
    n = len(m)
    pos = state[:3*n].reshape(n, 3)
    vel = state[3*n:].reshape(n, 3)
    diff = pos[None, :, :] - pos[:, None, :]          # diff[i,j] = pos_j - pos_i
    inv3 = ((diff**2).sum(-1) + eps**2) ** -1.5
    np.fill_diagonal(inv3, 0)
    acc = G * (diff * (m[None, :, None] * inv3[:, :, None])).sum(1)
    return np.concatenate([vel.ravel(), acc.ravel()])

def simulate(pos0, vel0, m, T, n_frames=500):
    m = np.asarray(m, float)
    y0 = np.concatenate([np.ravel(pos0), np.ravel(vel0)])
    sol = solve_ivp(nbody_rhs, (0, T), y0, args=(m,), method='DOP853',
                    t_eval=np.linspace(0, T, n_frames), rtol=1e-10, atol=1e-10)
    n = len(m)
    return sol.y[:3*n].T.reshape(-1, n, 3)            # (frames, bodies, xyz)

def animate(traj, names, colors, sizes, trail=80, frame_ms=30, title=""):
    nf, n, _ = traj.shape
    lim = np.abs(traj).max() * 1.15
    rng = np.random.default_rng(1)
    stars = rng.uniform(-lim, lim, (700, 3))

    base = [go.Scatter3d(x=stars[:,0], y=stars[:,1], z=stars[:,2], mode='markers',
                         marker=dict(size=1.2, color='white', opacity=0.7),
                         hoverinfo='skip', showlegend=False)]
    for i in range(n):   # trails
        base.append(go.Scatter3d(x=[], y=[], z=[], mode='lines',
                                 line=dict(color=colors[i], width=3), showlegend=False))
    for i in range(n):   # bodies
        base.append(go.Scatter3d(x=[traj[0,i,0]], y=[traj[0,i,1]], z=[traj[0,i,2]],
                                 mode='markers', name=names[i],
                                 marker=dict(size=sizes[i], color=colors[i],
                                             line=dict(color='white', width=0.5))))

    frames = []
    for k in range(nf):
        lo = max(0, k - trail)
        data  = [go.Scatter3d(x=traj[lo:k+1,i,0], y=traj[lo:k+1,i,1], z=traj[lo:k+1,i,2]) for i in range(n)]
        data += [go.Scatter3d(x=[traj[k,i,0]], y=[traj[k,i,1]], z=[traj[k,i,2]]) for i in range(n)]
        frames.append(go.Frame(data=data, traces=list(range(1, 2*n+1)), name=str(k)))

    ax = dict(visible=False, range=[-lim, lim])
    fig = go.Figure(data=base, frames=frames)
    fig.update_layout(
        title=dict(text=title, font=dict(color='white')),
        paper_bgcolor='black', height=750,
        scene=dict(xaxis=ax, yaxis=ax, zaxis=ax, bgcolor='black', aspectmode='cube'),
        legend=dict(font=dict(color='white')),
        updatemenus=[dict(type='buttons', x=0.05, y=0.05, buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=frame_ms, redraw=True), fromcurrent=True)]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')])])],
        sliders=[dict(steps=[dict(method='animate', label='',
                                  args=[[str(k)], dict(frame=dict(duration=0, redraw=True), mode='immediate')])
                             for k in range(0, nf, 5)], x=0.2, len=0.75)])
    return fig

In [ ]:
pos0 = [[1, 2, 0], [-1, 0, 0], [0, 2, 0]]
vel0 = [[0.5, 0, 0], [0, 0.9, 0], [0.1, 0.4, 0]]
m = [1, 1, 1]

traj = simulate(pos0, vel0, m, T=10, n_frames=500)
fig = animate(traj, ['Star 1', 'Star 2', 'Star 3'],
              ['#ffd54f', '#4fc3f7', '#ff7043'], [14, 14, 14],
              title='Three-Body Problem')
fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# ---------- physics (scaled units, G = 1) ----------
G = 1.0
M_sun, M_earth, M_moon = 1000.0, 1.0, 0.0123
R_e, R_m = 100.0, 2.0                      # Earth-Sun, Moon-Earth distances

v_e = np.sqrt(G * (M_sun + M_earth) / R_e)
v_m = np.sqrt(G * M_earth / R_m)

pos0 = np.array([[0, 0, 0], [R_e, 0, 0], [R_e + R_m, 0, 0]], float)
vel0 = np.array([[0, 0, 0], [0, v_e, 0], [0, v_e + v_m, 0]], float)
m = np.array([M_sun, M_earth, M_moon])
vel0 -= (m[:, None] * vel0).sum(0) / m.sum()      # keep system centred

traj = simulate(pos0, vel0, m, T=200, n_frames=900)   # ~1 "year", ~11 Moon orbits

# ---------- 2D animator that fills the screen ----------
def space_animation(traj2d, names, layers, trails, xr, yr, title, height=800, frame_ms=20):
    """layers[i] = list of (size, color, opacity) drawn on top of each other (for glow)."""
    nf, n, _ = traj2d.shape
    rng = np.random.default_rng(3)
    sx = rng.uniform(*xr, 900); sy = rng.uniform(*yr, 900)
    ss = rng.choice([1, 1.5, 2, 3], 900, p=[.5, .3, .15, .05])

    data = [go.Scatter(x=sx, y=sy, mode='markers', hoverinfo='skip', showlegend=False,
                       marker=dict(size=ss, color='white', opacity=0.75))]
    for i in range(n):                                            # trails
        data.append(go.Scatter(x=[], y=[], mode='lines', hoverinfo='skip', showlegend=False,
                               line=dict(color=layers[i][-1][1], width=2)))
    owner = []
    for i in range(n):                                            # bodies (+ glow layers)
        for j, (size, color, op) in enumerate(layers[i]):
            owner.append(i)
            data.append(go.Scatter(x=[traj2d[0, i, 0]], y=[traj2d[0, i, 1]], mode='markers',
                                   name=names[i], showlegend=(j == len(layers[i]) - 1),
                                   hoverinfo='skip',
                                   marker=dict(size=size, color=color, opacity=op)))

    frames = []
    for k in range(nf):
        d = []
        for i in range(n):
            lo = max(0, k - trails[i])
            d.append(go.Scatter(x=traj2d[lo:k+1, i, 0], y=traj2d[lo:k+1, i, 1]))
        for i in owner:
            d.append(go.Scatter(x=[traj2d[k, i, 0]], y=[traj2d[k, i, 1]]))
        frames.append(go.Frame(data=d, traces=list(range(1, len(data))), name=str(k)))

    fig = go.Figure(data=data, frames=frames)
    fig.update_layout(
        title=dict(text=title, font=dict(color='white', size=20), x=0.5),
        paper_bgcolor='black', plot_bgcolor='black', height=height,
        margin=dict(l=0, r=0, t=50, b=0),
        xaxis=dict(range=list(xr), visible=False, constrain='domain'),
        yaxis=dict(range=list(yr), visible=False, scaleanchor='x', scaleratio=1),
        legend=dict(font=dict(color='white', size=14), x=0.01, y=0.99, bgcolor='rgba(0,0,0,0)'),
        updatemenus=[dict(type='buttons', x=0.02, y=0.03, bgcolor='#222', font=dict(color='white'),
            buttons=[
              dict(label='▶ Play', method='animate',
                   args=[None, dict(frame=dict(duration=frame_ms, redraw=False),
                                    fromcurrent=True, transition=dict(duration=0))]),
              dict(label='⏸ Pause', method='animate',
                   args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')])])],
        sliders=[dict(x=0.15, len=0.8, y=0.03, font=dict(color='white'),
                      steps=[dict(method='animate', label='',
                                  args=[[str(k)], dict(frame=dict(duration=0, redraw=False),
                                                       mode='immediate')])
                             for k in range(0, nf, 5)])])
    return fig

# ---------- View 1: whole system (wide, fills the screen) ----------
EX = 7                                         # exaggerate Moon distance so it's visible
disp = traj.copy()
disp[:, 2] = traj[:, 1] + EX * (traj[:, 2] - traj[:, 1])

sun_layers   = [(120, '#ff6f00', 0.10), (85, '#ff9800', 0.25), (55, '#ffc107', 0.6), (38, '#fff59d', 1)]
earth_layers = [(26, '#1565c0', 0.25), (16, '#42a5f5', 1)]
moon_layers  = [(9, '#eceff1', 1)]

fig1 = space_animation(disp[:, :, :2], ['Sun', 'Earth', 'Moon (distance ×7)'],
                       [sun_layers, earth_layers, moon_layers],
                       trails=[1, 300, 120],
                       xr=(-215, 215), yr=(-122, 122),
                       title='Sun – Earth – Moon')
fig1.show()

# ---------- View 2: close-up, Earth-centred, true Moon distance ----------
rel = traj - traj[:, [1], :]
fig2 = space_animation(rel[:, 1:, :2], ['Earth', 'Moon'],
                       [[(70, '#1565c0', 0.25), (45, '#42a5f5', 1)],
                        [(18, '#eceff1', 1)]],
                       trails=[1, 150],
                       xr=(-3.3, 3.3), yr=(-3.3, 3.3),
                       title='Moon orbiting Earth (close-up)', height=800)
fig2.show()

In [ ]:
import numpy as np

# ---------- physics (scaled units, G = 1) ----------
G = 1.0
M_sun, M_venus, M_earth, M_moon = 1000.0, 0.815, 1.0, 0.0123
R_v, R_e, R_m = 72.0, 100.0, 2.0            # Venus-Sun, Earth-Sun, Moon-Earth

# Sun drift velocity (the whole system moves with it)
SUN_VEL   = np.array([0.35, 0.12, 0.0])
START_POS = np.array([-42.0, -14.0, 0.0])   # start left/below so it drifts across the screen

# Circular-orbit speeds
v_v = np.sqrt(G * (M_sun + M_venus) / R_v)
v_e = np.sqrt(G * (M_sun + M_earth) / R_e)
v_m = np.sqrt(G * M_earth / R_m)

# Venus starts at 120 degrees so it is not lined up with Earth
th = np.radians(120)
venus_pos = R_v * np.array([np.cos(th), np.sin(th), 0])
venus_vel = v_v * np.array([-np.sin(th), np.cos(th), 0])

#                 Sun        Venus       Earth            Moon
pos0 = np.array([[0, 0, 0], venus_pos, [R_e, 0, 0],      [R_e + R_m, 0, 0]], float)
vel0 = np.array([[0, 0, 0], venus_vel, [0, v_e, 0],      [0, v_e + v_m, 0]], float)
m    = np.array([M_sun, M_venus, M_earth, M_moon])

# 1) remove the system's own momentum, 2) add the drift to everyone, 3) shift start position
vel0 -= (m[:, None] * vel0).sum(0) / m.sum()
vel0 += SUN_VEL
pos0 += START_POS

traj = simulate(pos0, vel0, m, T=240, n_frames=1000)   # ~2 Venus orbits, ~1.2 Earth orbits

# ---------- View 1: whole system ----------
EX = 7                                         # exaggerate Moon distance so it is visible
disp = traj.copy()
disp[:, 3] = traj[:, 2] + EX * (traj[:, 3] - traj[:, 2])

sun_layers   = [(120, '#ff6f00', 0.10), (85, '#ff9800', 0.25), (55, '#ffc107', 0.6), (38, '#fff59d', 1)]
venus_layers = [(24, '#e65100', 0.20), (15, '#ffb74d', 1)]
earth_layers = [(26, '#1565c0', 0.25), (16, '#42a5f5', 1)]
moon_layers  = [(9, '#eceff1', 1)]

fig1 = space_animation(disp[:, :, :2],
                       ['Sun', 'Venus', 'Earth', 'Moon (distance ×7)'],
                       [sun_layers, venus_layers, earth_layers, moon_layers],
                       trails=[200, 220, 300, 120],     # Sun trail shows its drift path
                       xr=(-240, 240), yr=(-135, 135),
                       title='Sun – Venus – Earth – Moon (Sun drifting)')
fig1.show()

# ---------- View 2: Earth-centred close-up, true Moon distance ----------
rel = traj - traj[:, [2], :]
fig2 = space_animation(rel[:, 2:4, :2], ['Earth', 'Moon'],
                       [[(70, '#1565c0', 0.25), (45, '#42a5f5', 1)],
                        [(18, '#eceff1', 1)]],
                       trails=[1, 150],
                       xr=(-3.3, 3.3), yr=(-3.3, 3.3),
                       title='Moon orbiting Earth (close-up)')
fig2.show()

In [ ]:
fig1.write_html("sun_venus_earth_moon.html", include_plotlyjs=True, auto_play=False)

In [ ]:
import subprocess, sys, os
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imageio", "imageio-ffmpeg", "matplotlib"])

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imageio.v2 as imageio

def save_video_from_fig(fig, filename, step=2, fps=30, w=1280):
    xr, yr = list(fig.layout.xaxis.range), list(fig.layout.yaxis.range)
    h = int(round(w * (yr[1] - yr[0]) / (xr[1] - xr[0]) / 2) * 2)   # keep the same shape as the animation
    dpi = 100

    fm = plt.figure(figsize=(w / dpi, h / dpi), dpi=dpi, facecolor="black")
    ax = fm.add_axes([0, 0, 1, 1])
    ax.set_facecolor("black"); ax.axis("off")
    ax.set_xlim(xr); ax.set_ylim(yr)

    # background stars
    st = fig.data[0]
    ax.scatter(st.x, st.y, s=(np.array(st.marker.size, float) * 0.7) ** 2,
               c="white", alpha=0.75, linewidths=0, zorder=0)

    # trails and bodies (same order as in the Plotly figure)
    artists, legend_handles = {}, []
    for idx in range(1, len(fig.data)):
        tr = fig.data[idx]
        if tr.mode == "lines":
            (a,) = ax.plot([], [], color=tr.line.color, lw=1.6, zorder=idx)
        else:
            (a,) = ax.plot([], [], "o", color=tr.marker.color, alpha=tr.marker.opacity,
                           markersize=tr.marker.size * 0.72, markeredgewidth=0, zorder=idx)
            if tr.showlegend:
                legend_handles.append((a, tr.name))
        artists[idx] = a

    leg = ax.legend([a for a, _ in legend_handles], [n for _, n in legend_handles],
                    loc="upper left", frameon=False, labelcolor="white", fontsize=14)
    for a, _ in legend_handles:
        pass
    ax.set_title(fig.layout.title.text or "", color="white", fontsize=20, pad=-30)

    writer = imageio.get_writer(filename, fps=fps, codec="libx264", quality=8, macro_block_size=None)
    frames = range(0, len(fig.frames), step)
    for n, k in enumerate(frames):
        fr = fig.frames[k]
        for idx, d in zip(fr.traces, fr.data):
            artists[idx].set_data(d.x, d.y)
        fm.canvas.draw()
        writer.append_data(np.asarray(fm.canvas.buffer_rgba())[:, :, :3])
        if n % 50 == 0:
            print(f"Rendering {n}/{len(frames)}")
    writer.close()
    plt.close(fm)
    print("Saved:", os.path.abspath(filename))

save_video_from_fig(fig1, "sun_venus_earth_moon.mp4")